In [1]:
import numpy as np
import numpy.typing as npt
from pathlib import Path
from astropy.time import Time
from sorts import equidistant_sampling
from sorts.interpolation import Legendre8, Linear
from sorts.population import master_catalog, master_catalog_factor
from sorts.propagator import SGP4
from sorts.space_object import SpaceObject
from sorts.radar.tx_rx import Station
from sorts.radar.radars import get_radar
from sorts.types import Datetime64_us, Timedelta64_us, Float64_as_sec
from sorts.utils import to_datetime64_us, to_pydatetime
from sorts.controller_v2.tracker_controller import TrackerController
from sorts.controller_v2.fence_scan_controller_new import FenceScanController
from sorts.schedule_v2 import Schedule, ExperimentDetail
from sorts.scheduler_v2.priority_scheduling import priority_scheduling
from sorts.simulation_v2 import StxMrxSimulation, StxMrxSimulationParam, Observation

# import for plottings
from IPython.display import display
import pandas as pd
import ipywidgets as widgets
from sorts.plotting_deps import lp, bp, bokeh_models, pn
from sorts import plots

The geodata is provided by © OpenStreetMap contributors and is made available here under the Open Database License (ODbL).


In [2]:
# disable pandas table wrapping
pd.set_option("display.expand_frame_repr", False)

# activate Bokeh output in Jupyter notebook
from bokeh.io import output_notebook, push_notebook
output_notebook()

# config and init lets-plot
lp.LetsPlot.setup_html()

# config and init panel
pn.extension("tabulator", comms='ipywidgets',
    # sizing_mode="stretch_width",
)

Loading BokehJS ...

In [3]:
epoch = Time(53005.0, format="mjd", scale="utc")  # 2004-01-01 00:00:00Z

# the first set of value used, not much use now; kept for ref
# start_time = Time("2025-06-30 00:00:00")
# end_time = Time("2025-06-30 00:00:01")
# control_slice_duration = np.timedelta64(10_000, "us")  # 10ms

# a 1 sec long period, the sbobj should be very close to right up ahead of eiscat3d tx-0 station
# start_time = Time("2025-01-01 04:04:00")
# end_time = Time("2025-01-01 04:04:01")
# control_slice_duration = np.timedelta64(10_000, "us")  # 10ms

# an extended duration which expands around from the 1 sec period above
# the `control_slice_duration` is much longer than normal, practical radar `control_slice_duration`
# for easier debugging, inspection of scheduling/schedules
start_time = Time("2025-01-01 02:45:00")
end_time = Time("2025-01-01 06:15:00")
control_slice_duration = np.timedelta64(int(60 * 1e6), "us")

# same as above, but use more realistic 10ms `control_slice_duration`
# start_time = Time("2025-01-01 02:45:00")
# end_time = Time("2025-01-01 06:15:00")
# control_slice_duration = np.timedelta64(10_000, "us")  # 10ms

eiscat3d = get_radar("eiscat3d", "stage1-array")

tracked_spobj = SpaceObject(
    oid=-1,
    propagator=SGP4,
    propagator_options={"settings": {"out_frame": "ITRF"}},
    a=7200e3,
    e=0.02,
    i=75,
    raan=86,
    aop=0,
    mu0=60,
    epoch=epoch,
    parameters={"d": 0.1},
)

catalog_fpath = Path() / ".." /  ".." / "local_data" / "celn_20090501_00.sim"
_spobj_pop = master_catalog(
    catalog_fpath,
    propagator=SGP4,
    propagator_options={"settings": {"in_frame": "TEME", "out_frame": "ITRF"}},
)
rand_seed = 120389
# TODO: reduce the filter size to more sensible value
spobj_pop = master_catalog_factor(_spobj_pop, treshhold=5.0, seed=rand_seed)
spobjs = [tracked_spobj, *[spobj_pop.get_object(i) for i in range(spobj_pop.shape[0])]]

exp_detail_0 = ExperimentDetail(
    id=0,
    coh_int_bandwidth=1.0,
    ipp=1.0,
    pulse_length=1.0,
    power=5000000.0,
    bandwidth=52.08333333333333,
    duty_cycle=1.0,
    noise_temp=150.0,
    slice_duration=control_slice_duration
)

exp_detail_1 = ExperimentDetail(
    id=1,
    coh_int_bandwidth=1.0,
    ipp=1.0,
    pulse_length=1.0,
    power=5000000.0,
    bandwidth=52.08333333333333,
    duty_cycle=1.0,
    noise_temp=150.0,
    slice_duration=control_slice_duration
)

exp_detail_map = {0: exp_detail_0, 1: exp_detail_1}

time_arr: npt.NDArray[Datetime64_us] = np.arange(
    to_datetime64_us(start_time),
    to_datetime64_us(end_time),
    control_slice_duration,
)
# time_arr = time_arr[::4] # TODO: remove; strided to bring up the effects of scheduling
dt_arr: npt.NDArray[Timedelta64_us] = time_arr - to_datetime64_us(epoch)
dsec_arr: npt.NDArray[Float64_as_sec] = dt_arr.astype(np.float64) / 1e6  # type: ignore

ecefs = tracked_spobj.get_state(dsec_arr)

trackerController = TrackerController(
    tx_station=eiscat3d.tx[0],
    # rx_stations=eiscat3d.rx[0:1],
    rx_stations=eiscat3d.rx[0:2],
    time=time_arr,
    space_object_states=ecefs,
    exp_detail=exp_detail_0,
    min_elevation=10,
)

fenceScanController = FenceScanController(
    tx_station=eiscat3d.tx[0],
    # rx_stations=eiscat3d.rx[0:1],
    rx_stations=eiscat3d.rx[0:2],
    exp_datail=exp_detail_1,
    azimuth=90, # sweep from east to west
    min_elevation=30,
    pointings_per_cycle=40,
    # scan_range=np.linspace(300e3, 1000e3, num=10, dtype=np.float64),
    scan_range=np.array([300e3], dtype=np.float64),
)

In [4]:
data_table = plots.space_object_population_table_plot(spobj_pop)

bp.show(data_table)

In [5]:
plot = plots.kepler_space_object_on_map(spobjs[0], epoch, start_time=start_time, end_time=end_time)
plot.show()

In [6]:
# plot = plots.kepler_space_object_on_map(spobjs[4], epoch, start_time=start_time, end_time=end_time) # interesting s shape
# plot = plots.kepler_space_object_on_map(spobjs[5], epoch, start_time=start_time, end_time=end_time) # show be visible to eiscat
plot = plots.kepler_space_object_on_map(spobjs[17], epoch, start_time=start_time, end_time=end_time) # show be visible to eiscat
plot.show()

/home/tszhinh/projects/sorts/.venv/lib/python3.10/site-packages/erfa/core.py:133: ErfaWarning: ERFA function "taiutc" yielded 500 of "dubious year (Note 4)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)
/home/tszhinh/projects/sorts/.venv/lib/python3.10/site-packages/erfa/core.py:133: ErfaWarning: ERFA function "utcut1" yielded 500 of "dubious year (Note 3)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)
/home/tszhinh/projects/sorts/.venv/lib/python3.10/site-packages/erfa/core.py:133: ErfaWarning: ERFA function "utctai" yielded 500 of "dubious year (Note 3)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)


In [7]:
# plots.ecef_states_positions_plot(ecefs)

In [8]:
tracker_schs = trackerController.generate()
tracker_tx_sch_df = tracker_schs.tx_schedule.to_dataframe()
tracker_tx_sch_df

,start_time,pointing_az,pointing_el,exp_num,end_time
0,2025-01-01 02:45:00,17.942607,14.412365,0,2025-01-01 02:46:00
1,2025-01-01 02:46:00,17.521616,11.088799,0,2025-01-01 02:47:00
2,2025-01-01 03:44:00,-152.400713,13.468171,0,2025-01-01 03:45:00
3,2025-01-01 03:45:00,-152.513503,17.085825,0,2025-01-01 03:46:00
4,2025-01-01 03:46:00,-152.629369,20.698738,0,2025-01-01 03:47:00
...,...,...,...,...,...
89,2025-01-01 06:08:00,61.132328,25.656604,0,2025-01-01 06:09:00
90,2025-01-01 06:09:00,61.595549,22.334598,0,2025-01-01 06:10:00
91,2025-01-01 06:10:00,62.053178,19.012611,0,2025-01-01 06:11:00
92,2025-01-01 06:11:00,62.507070,15.690203,0,2025-01-01 06:12:00


In [9]:
# check schedule df memory usage (MB)
tracker_tx_sch_df.memory_usage().sum()/1e6

np.float64(0.003888)

In [10]:
fence_schs = fenceScanController.generate(start_time, end_time)
fence_tx_sch_df = fence_schs.tx_schedule.to_dataframe()
fence_tx_sch_df

,start_time,pointing_az,pointing_el,exp_num,end_time
0,2025-01-01 02:45:00,90.0,30.000000,1,2025-01-01 02:46:00
1,2025-01-01 02:46:00,90.0,33.076923,1,2025-01-01 02:47:00
2,2025-01-01 02:47:00,90.0,36.153846,1,2025-01-01 02:48:00
3,2025-01-01 02:48:00,90.0,39.230769,1,2025-01-01 02:49:00
4,2025-01-01 02:49:00,90.0,42.307692,1,2025-01-01 02:50:00
...,...,...,...,...,...
205,2025-01-01 06:10:00,90.0,45.384615,1,2025-01-01 06:11:00
206,2025-01-01 06:11:00,90.0,48.461538,1,2025-01-01 06:12:00
207,2025-01-01 06:12:00,90.0,51.538462,1,2025-01-01 06:13:00
208,2025-01-01 06:13:00,90.0,54.615385,1,2025-01-01 06:14:00


In [11]:
# check schedule df memory usage (MB)
fence_tx_sch_df.memory_usage().sum()/1e6

np.float64(0.008528)

In [12]:
tx_sch = tracker_schs.tx_schedule
# plots.azel_polar_plot(tx_sch.pointing_az, tx_sch.pointing_el)

In [13]:
plots.azel_polar_plot(fence_schs.tx_schedule.pointing_az, fence_schs.tx_schedule.pointing_el)

In [14]:
plots.azel_polar_plot(fence_schs.rx_schedules[0].pointing_az, fence_schs.rx_schedules[0].pointing_el)

In [15]:
(tracker_schs.tx_schedule.meta, fence_schs.tx_schedule.meta)

({0: ExperimentDetail(id=0, coh_int_bandwidth=1.0, ipp=1.0, pulse_length=1.0, power=5000000.0, bandwidth=52.08333333333333, duty_cycle=1.0, noise_temp=150.0, slice_duration=np.timedelta64(60000000,'us'))},
 {1: ExperimentDetail(id=1, coh_int_bandwidth=1.0, ipp=1.0, pulse_length=1.0, power=5000000.0, bandwidth=52.08333333333333, duty_cycle=1.0, noise_temp=150.0, slice_duration=np.timedelta64(60000000,'us'))})

In [16]:
master_sch = priority_scheduling([tracker_schs.tx_schedule, fence_schs.tx_schedule], exp_detail_map)
master_sch

Schedule(meta={0: ExperimentDetail(id=0, coh_int_bandwidth=1.0, ipp=1.0, pulse_length=1.0, power=5000000.0, bandwidth=52.08333333333333, duty_cycle=1.0, noise_temp=150.0, slice_duration=np.timedelta64(60000000,'us')), 1: ExperimentDetail(id=1, coh_int_bandwidth=1.0, ipp=1.0, pulse_length=1.0, power=5000000.0, bandwidth=52.08333333333333, duty_cycle=1.0, noise_temp=150.0, slice_duration=np.timedelta64(60000000,'us'))}, start_time=array(['2025-01-01T02:45:00.000000', '2025-01-01T02:46:00.000000',
       '2025-01-01T02:48:00.000000', '2025-01-01T02:49:00.000000',
       '2025-01-01T02:50:00.000000', '2025-01-01T02:51:00.000000',
       '2025-01-01T02:52:00.000000', '2025-01-01T02:53:00.000000',
       '2025-01-01T02:54:00.000000', '2025-01-01T02:55:00.000000',
       '2025-01-01T02:56:00.000000', '2025-01-01T02:57:00.000000',
       '2025-01-01T02:58:00.000000', '2025-01-01T02:59:00.000000',
       '2025-01-01T03:00:00.000000', '2025-01-01T03:01:00.000000',
       '2025-01-01T03:02:00.000

In [17]:
master_sch_df = master_sch.to_dataframe()
master_sch_df

,start_time,pointing_az,pointing_el,exp_num,end_time
0,2025-01-01 02:45:00,17.942607,14.412365,0,2025-01-01 02:46:00
1,2025-01-01 02:46:00,17.521616,11.088799,0,2025-01-01 02:47:00
2,2025-01-01 02:48:00,90.000000,39.230769,1,2025-01-01 02:49:00
3,2025-01-01 02:49:00,90.000000,42.307692,1,2025-01-01 02:50:00
4,2025-01-01 02:50:00,90.000000,45.384615,1,2025-01-01 02:51:00
...,...,...,...,...,...
199,2025-01-01 06:08:00,61.132328,25.656604,0,2025-01-01 06:09:00
200,2025-01-01 06:09:00,61.595549,22.334598,0,2025-01-01 06:10:00
201,2025-01-01 06:10:00,62.053178,19.012611,0,2025-01-01 06:11:00
202,2025-01-01 06:11:00,62.507070,15.690203,0,2025-01-01 06:12:00


In [18]:
master_sch_df[master_sch_df["exp_num"] == 1]

,start_time,pointing_az,pointing_el,exp_num,end_time
2,2025-01-01 02:48:00,90.0,39.230769,1,2025-01-01 02:49:00
3,2025-01-01 02:49:00,90.0,42.307692,1,2025-01-01 02:50:00
4,2025-01-01 02:50:00,90.0,45.384615,1,2025-01-01 02:51:00
5,2025-01-01 02:51:00,90.0,48.461538,1,2025-01-01 02:52:00
6,2025-01-01 02:52:00,90.0,51.538462,1,2025-01-01 02:53:00
...,...,...,...,...,...
153,2025-01-01 05:21:00,270.0,39.230769,1,2025-01-01 05:22:00
154,2025-01-01 05:22:00,270.0,36.153846,1,2025-01-01 05:23:00
155,2025-01-01 05:23:00,270.0,33.076923,1,2025-01-01 05:24:00
156,2025-01-01 05:24:00,270.0,30.000000,1,2025-01-01 05:25:00


In [19]:
# plots.schedule_plot(master_sch)

In [20]:
schedule = master_sch

df = schedule.to_dataframe()

start_datetime_widget = widgets.DatetimePicker(
    value=df[schedule.cn.start_time].min().tz_localize("utc"),
    description='Start Time',
)
end_datetime_widget = widgets.DatetimePicker(
    value=df[schedule.cn.start_time].min().tz_localize("utc") + np.timedelta64(5, "m"),
    description='End Time',
)

date_range_widget = widgets.HBox([start_datetime_widget, end_datetime_widget])
date_range_widget

In [21]:
# TODO: leverage `notebook_handle`, e.g. `plot_nbh = bp.show(plot, notebook_handle=True)` ?
plot = plots.schedule_plot_bokeh(master_sch, start_datetime_widget.value.replace(tzinfo=None), end_datetime_widget.value.replace(tzinfo=None))
bp.show(plot)

In [22]:
tx_station:Station = eiscat3d.tx[0]
tx_station.uid = ("eiscat3d", "stage1-array", "tx", "0")
rx_station:Station = eiscat3d.rx[0]
rx_station.uid = ("eiscat3d", "stage1-array", "rx", "0")

sim = StxMrxSimulation(
    StxMrxSimulationParam(
        tx_station=tx_station,
        tx_schedule=master_sch,
        rx_stations=[rx_station],
        rx_schedules=[master_sch],
        exp_num_map=exp_detail_map,
        # TODO: chg StxMrxSimulationParam to take Datetime_like for datetime params
        epoch=to_pydatetime(epoch),
        start_time=to_pydatetime(start_time),
        end_time=to_pydatetime(end_time),
        space_objects=[o for i, o in enumerate(spobjs) if i in [0, 4, 5, 17]], # just picked a few from the whole list for now
        space_objects_dt_sampler_s=lambda orbit, start_time, end_time: equidistant_sampling(
            orbit=orbit,
            start_t=(to_pydatetime(start_time) - to_pydatetime(epoch)).total_seconds(),
            end_t=(to_pydatetime(end_time) - to_pydatetime(epoch)).total_seconds(),
            max_dpos=1e3,
        ),
        space_objects_dt_interpolator_s=Linear,
    )
)

In [23]:
obss = sim.calculate_observations()
spobjs_states_interps = sim._spobjs_states_interps

/home/tszhinh/projects/sorts/.venv/lib/python3.10/site-packages/erfa/core.py:133: ErfaWarning: ERFA function "taiutc" yielded 61143 of "dubious year (Note 4)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)
/home/tszhinh/projects/sorts/.venv/lib/python3.10/site-packages/erfa/core.py:133: ErfaWarning: ERFA function "utcut1" yielded 61143 of "dubious year (Note 3)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)
/home/tszhinh/projects/sorts/.venv/lib/python3.10/site-packages/erfa/core.py:133: ErfaWarning: ERFA function "utctai" yielded 61143 of "dubious year (Note 3)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)
/home/tszhinh/projects/sorts/.venv/lib/python3.10/site-packages/erfa/core.py:133: ErfaWarning: ERFA function "taiutc" yielded 94960 of "dubious year (Note 4)"
  warn(f'ERFA function "{func_name}" yielded {wmsg}', ErfaWarning)
/home/tszhinh/projects/sorts/.venv/lib/python3.10/site-packages/erfa/core.py:133: ErfaWarning: E

In [24]:
obss[0]

Observation(id="0-(np.datetime64('2025-01-01T04:02:46.812565'), np.datetime64('2025-01-01T04:08:25.378784'))", passage=Passage(space_object=<sorts.space_object.SpaceObject object at 0x777d8ae0f850>, tx_station=<sorts.radar.tx_rx.TX object at 0x777d8ae0f7f0>, rx_station=<sorts.radar.tx_rx.RX object at 0x777d8ae0f670>, epoch=np.datetime64('2004-01-01T00:00:00.000000'), time_range=(np.datetime64('2025-01-01T04:02:46.812565'), np.datetime64('2025-01-01T04:08:25.378784'))), snr=array([7.63255463e-05, 1.47614348e-03, 9.04843374e-03, 1.25642468e-03,
       7.69899684e-03, 5.62814879e-03]), range=array([2831576.05212596, 2314893.67540699, 2011130.07566348,
       2018681.55086723, 2333385.28935571, 2853660.7970656 ]), range_rx=array([1415788.02606298, 1157446.8377035 , 1005565.03783174,
       1009340.77543362, 1166692.64467785, 1426830.3985328 ]), range_rate=array([1., 1., 1., 1., 1., 1.]), tx_k=array([[-0.13201888,  0.03804332,  0.27611875,  0.50841633,  0.64263748,
         0.69148281],
   

In [25]:
obss_df = Observation.list_to_dataframe(obss)
obss_df

,space_object_id,tx_station_id,rx_station_id,epoch,time_range,snr,range,range_rx,range_rate,tx_k,rx_k
0,-1,"(eiscat3d, stage1-array, tx, 0)","(eiscat3d, stage1-array, rx, 0)",2004-01-01,"(2025-01-01T04:02:46.812565, 2025-01-01T04:08:...","[7.63255463071037e-05, 0.0014761434837294398, ...","[2831576.052125955, 2314893.6754069906, 201113...","[1415788.0260629775, 1157446.8377034953, 10055...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0]","[[-0.13201887541133717, 0.03804331926450933, 0...","[[-0.13201887541133717, 0.03804331926450933, 0..."
1,-1,"(eiscat3d, stage1-array, tx, 0)","(eiscat3d, stage1-array, rx, 0)",2004-01-01,"(2025-01-01T05:45:56.250950, 2025-01-01T05:51:...","[0.00014833990386244771, 0.0002747921937761362...","[2970227.287985119, 2401913.877205393, 2007936...","[1485113.6439925595, 1200956.9386026964, 10039...","[1.0, 1.0, 1.0, 1.0, 1.0, 1.0]","[[-0.815140306928066, -0.7113784531175842, -0....","[[-0.815140306928066, -0.7113784531175842, -0...."
2,4,"(eiscat3d, stage1-array, tx, 0)","(eiscat3d, stage1-array, rx, 0)",2004-01-01,"(2025-01-01T03:25:41.982013, 2025-01-01T03:30:...","[225.39588416930485, 0.5788132271741752, 15413...","[2315372.276490303, 1738832.3207809622, 143858...","[1157686.1382451516, 869416.1603904811, 719291...","[1.0, 1.0, 1.0, 1.0, 1.0]","[[-0.7889173270934465, -0.581805661461436, -0....","[[-0.7889173270934465, -0.581805661461436, -0...."
3,4,"(eiscat3d, stage1-array, tx, 0)","(eiscat3d, stage1-array, rx, 0)",2004-01-01,"(2025-01-01T05:06:30.412143, 2025-01-01T05:11:...","[6719.633436400606, 9597.855724885661, 2732.65...","[2196477.223617172, 1677237.6547763208, 147708...","[1098238.611808586, 838618.8273881604, 738542....","[1.0, 1.0, 1.0, 1.0, 1.0]","[[-0.7652322738146587, -0.4812058929727864, 0....","[[-0.7652322738146587, -0.4812058929727864, 0...."
4,16,"(eiscat3d, stage1-array, tx, 0)","(eiscat3d, stage1-array, rx, 0)",2004-01-01,"(2025-01-01T04:09:08.317692, 2025-01-01T04:14:...","[10684.570763445336, 3942.582140882196, 101.69...","[2436641.4709606147, 2132891.8186298655, 21368...","[1218320.7354803074, 1066445.9093149328, 10684...","[1.0, 1.0, 1.0, 1.0, 1.0]","[[0.5991557418818109, 0.6357519354195282, 0.58...","[[0.5991557418818109, 0.6357519354195282, 0.58..."
5,16,"(eiscat3d, stage1-array, tx, 0)","(eiscat3d, stage1-array, rx, 0)",2004-01-01,"(2025-01-01T05:49:24.433925, 2025-01-01T05:54:...","[22.408272699063403, 3762.1004677504347, 5861....","[2487644.7776287226, 2019722.9562591473, 18466...","[1243822.3888143613, 1009861.4781295736, 92334...","[1.0, 1.0, 1.0, 1.0, 1.0]","[[0.0886664676786372, -0.11585222596036557, -0...","[[0.0886664676786372, -0.11585222596036557, -0..."
